In [13]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

Created a chunk of size 470, which is longer than the specified 100
Created a chunk of size 347, which is longer than the specified 100
Created a chunk of size 670, which is longer than the specified 100
Created a chunk of size 984, which is longer than the specified 100
Created a chunk of size 791, which is longer than the specified 100


In [14]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
 Document(metadata={'source': 'speech.txt'}, page_content

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(docs, embeddings)
db

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8859.27it/s]


In [17]:
### querying
query = "What does the speaker believe is the main reason the United States should enter the war?"
docs = db.similarity_search(query)
docs[0].page_content

'Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'

In [19]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

'Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'

In [20]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='dd29c0a4-bff9-4097-a58a-949301ed7a10', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
  np.float32(1.2606809)),
 (Document(id='5e92e042-0da3-48e2-bbce-bbed2c14f1b4', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fi

In [21]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[-0.008726337924599648,
 -0.018167581409215927,
 0.0051252711564302444,
 -0.014269954524934292,
 0.0140144731849432,
 0.07833600044250488,
 0.08270474523305893,
 -0.08437390625476837,
 -0.0556509867310524,
 0.01933875121176243,
 -0.06483781337738037,
 0.06825461238622665,
 -0.01838250644505024,
 0.01082257553935051,
 0.04148862138390541,
 0.03414865955710411,
 0.022960972040891647,
 -0.10548100620508194,
 0.011171412654221058,
 0.01831481233239174,
 0.03937721997499466,
 -0.00464666960760951,
 0.03150429204106331,
 0.06253273040056229,
 -0.026338351890444756,
 0.03288443014025688,
 -0.0072478801012039185,
 0.0814603939652443,
 -0.05030946061015129,
 0.022972041741013527,
 0.0027099279686808586,
 0.003941667266190052,
 0.0334957055747509,
 0.03611496090888977,
 0.02576518803834915,
 -0.012924494221806526,
 0.10891018062829971,
 0.011238056235015392,
 -0.02451951801776886,
 -0.04274721071124077,
 -0.030726773664355278,
 -0.050319548696279526,
 0.04366016015410423,
 0.08062638342380524,
 

In [24]:
docs_and_score = db.similarity_search_with_score_by_vector(embedding_vector)
docs_and_score

[(Document(id='dd29c0a4-bff9-4097-a58a-949301ed7a10', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
  np.float32(1.2606809)),
 (Document(id='5e92e042-0da3-48e2-bbce-bbed2c14f1b4', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fi

In [25]:
###Saving and loading the vectorstore
db.save_local("faiss_index")

In [26]:
new_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
docs = new_db.similarity_search(query)

In [27]:
docs

[Document(id='dd29c0a4-bff9-4097-a58a-949301ed7a10', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
 Document(id='5e92e042-0da3-48e2-bbce-bbed2c14f1b4', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we 